# GARCH volatility across multiple horizons

Forecasts conditional volatility over 1, 5 and 21 trading days and ranks a configured asset universe.

**Status: exploratory research; not production validated.**

## Limitations
No out-of-sample forecast evaluation is implemented. The ranking measures volatility, not profitability or liquidity. Bands are multiples of volatility, not calibrated confidence intervals. Check optimizer convergence, residual diagnostics, completed bars and data quality before interpretation.

Original source: `Garchsemanal.ipynb`. Outputs cleared for publication. Analytical code preserved.


In [ ]:
# ============================================================
# RANK DE AÇÕES POR LIQUIDEZ + VOLATILIDADE
# FORECAST GARCH SEMANAL + PLOT DE PERNAS DE VOLATILIDADE
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tvDatafeed import TvDatafeed, Interval
from arch import arch_model

# ============================================================
# 1) CONFIGURAÇÕES
# ============================================================

TV_USERNAME = None
TV_PASSWORD = None

EXCHANGE = "BMFBOVESPA"

UNIVERSE = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3",
    "ABEV3", "WEGE3", "RENT3", "SUZB3", "PRIO3",
    "B3SA3", "GGBR4", "USIM5", "VBBR3", "CMIG4",
    "WIN1!", "WDO1!", "CSNA3", "SANB11", "SMTO3",
    "CSAN3",
]

# Ativo escolhido manualmente para o gráfico
# Ex.: "PETR4", "VALE3", "WIN1!", "WDO1!"
SELECTED_SYMBOL = "WDO1!"

# None = usa o top1 do ranking
# SELECTED_SYMBOL = None

N_BARS_DAILY = 4200   # histórico diário para GARCH
N_BARS_WEEKLY = 360   # histórico semanal para gráfico

TOP_N = 24

# pesos do score forward-looking
WEIGHT_D1 = 0.40
WEIGHT_W1 = 0.35
WEIGHT_M1 = 0.25

# pernas da vol da próxima semana
LEG_MULTIPLIERS = [0.50, 0.80, 1.00, 1.50]

# ============================================================
# 2) CONEXÃO
# ============================================================

def get_tv():
    try:
        if TV_USERNAME and TV_PASSWORD:
            return TvDatafeed(TV_USERNAME, TV_PASSWORD)
        return TvDatafeed()
    except Exception as e:
        raise RuntimeError(f"Erro ao iniciar TvDatafeed: {e}")

# ============================================================
# 3) DOWNLOAD COM RETRY
# ============================================================

def fetch_data(tv, symbol, interval, exchange=EXCHANGE, n_bars=300, retries=3, sleep_sec=2):
    for attempt in range(retries):
        try:
            df = tv.get_hist(
                symbol=symbol,
                exchange=exchange,
                interval=interval,
                n_bars=n_bars
            )

            if df is not None and not df.empty:
                df = df.copy()
                df.index = pd.to_datetime(df.index)
                df = df.sort_index()
                df.columns = [c.lower() for c in df.columns]

                required = {"open", "high", "low", "close", "volume"}
                if required.issubset(df.columns):
                    return df

        except Exception:
            pass

        time.sleep(sleep_sec)

    return None

# ============================================================
# 4) GARCH MULTI-HORIZONTE
# ============================================================

def fit_garch_multi_horizon(df_daily):
    """
    Ajusta GARCH(1,1) nos retornos diários e estima:
    - próximo dia (1 pregão)
    - próxima semana (~5 pregões)
    - próximo mês (~21 pregões)

    Retorna volatilidades em %.
    """
    if df_daily is None or df_daily.empty:
        return None

    ret = np.log(df_daily["close"] / df_daily["close"].shift(1)).dropna()

    # amostra mínima
    if len(ret) < 150:
        return None

    ret_pct = ret * 100.0

    try:
        model = arch_model(
            ret_pct,
            mean="Constant",
            vol="GARCH",
            p=1,
            q=1,
            dist="t"
        )

        res = model.fit(disp="off")
        fcst = res.forecast(horizon=21, reindex=False)

        variances = fcst.variance.values[-1]   # 21 variâncias previstas

        # previsão correta por horizonte acumulado
        garch_d1 = np.sqrt(variances[0])
        garch_w1 = np.sqrt(np.sum(variances[:5]))
        garch_m1 = np.sqrt(np.sum(variances[:21]))

        return {
            "garch_d1_%": float(garch_d1),
            "garch_w1_%": float(garch_w1),
            "garch_m1_%": float(garch_m1),
            "garch_w1_decimal": float(garch_w1 / 100.0),
            "model": res
        }

    except Exception:
        return None

# ============================================================
# 5) RANKING FORWARD-LOOKING
# ============================================================

def build_ranking(tv, universe):
    rows = []

    for symbol in universe:
        print(f"Processando {symbol}...")

        df_daily = fetch_data(
            tv, symbol, Interval.in_daily,
            exchange=EXCHANGE, n_bars=N_BARS_DAILY
        )

        if df_daily is None:
            print(f"[falhou] {symbol}")
            continue

        garch = fit_garch_multi_horizon(df_daily)
        if garch is None:
            print(f"[garch falhou] {symbol}")
            continue

        last_close = float(df_daily["close"].iloc[-1])

        rows.append({
            "symbol": symbol,
            "last_close": last_close,
            "garch_d1_%": garch["garch_d1_%"],
            "garch_w1_%": garch["garch_w1_%"],
            "garch_m1_%": garch["garch_m1_%"]
        })

    rank = pd.DataFrame(rows)

    if rank.empty:
        raise RuntimeError("Nenhum ativo retornou dados válidos.")

    rank["rank_d1"] = rank["garch_d1_%"].rank(pct=True)
    rank["rank_w1"] = rank["garch_w1_%"].rank(pct=True)
    rank["rank_m1"] = rank["garch_m1_%"].rank(pct=True)

    rank["score"] = (
        WEIGHT_D1 * rank["rank_d1"] +
        WEIGHT_W1 * rank["rank_w1"] +
        WEIGHT_M1 * rank["rank_m1"]
    )

    rank["perfil"] = np.where(
        (rank["garch_d1_%"] >= rank["garch_w1_%"] * 0.55) & (rank["garch_d1_%"] >= rank["garch_m1_%"] * 0.25),
        "Curto prazo",
        np.where(
            rank["garch_w1_%"] >= rank["garch_m1_%"] * 0.45,
            "Swing",
            "Posicional"
        )
    )

    rank = rank.sort_values(
        ["score", "garch_w1_%", "garch_d1_%"],
        ascending=False
    ).reset_index(drop=True)

    return rank

# ============================================================
# 6) PERNAS DE VOLATILIDADE
# ============================================================

def make_volatility_legs(last_close, expected_vol_decimal, multipliers=None):
    if multipliers is None:
        multipliers = [0.50, 0.80, 1.00, 1.50]

    legs = []
    for m in multipliers:
        pct_move = m * expected_vol_decimal
        legs.append({
            "label": f"{int(m * 100)}%",
            "pct_move": pct_move,
            "upper": last_close * (1 + pct_move),
            "lower": last_close * (1 - pct_move)
        })
    return legs

# ============================================================
# 7) PLOT
# ============================================================

def plot_asset_with_forecast(symbol, df_weekly, row_info, garch_info):
    last_close = df_weekly["close"].iloc[-1]
    exp_w1_pct = garch_info["garch_w1_%"]
    exp_w1_dec = garch_info["garch_w1_decimal"]

    legs = make_volatility_legs(last_close, exp_w1_dec, LEG_MULTIPLIERS)

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.plot(df_weekly.index, df_weekly["close"], linewidth=2, label=f"{symbol} - Close semanal")
    ax.axhline(last_close, linestyle="--", linewidth=1.2, label=f"Último fechamento: {last_close:.2f}")

    for leg in legs:
        ax.axhline(
            leg["upper"], linestyle=":", linewidth=1.4,
            label=f"+{leg['label']} = {leg['upper']:.2f}"
        )
        ax.axhline(
            leg["lower"], linestyle=":", linewidth=1.4,
            label=f"-{leg['label']} = {leg['lower']:.2f}"
        )

    title = (
        f"{symbol} | GARCH D+1: {row_info['garch_d1_%']:.2f}% | "
        f"W+1: {row_info['garch_w1_%']:.2f}% | "
        f"M+1: {row_info['garch_m1_%']:.2f}% | "
        f"Score: {row_info['score']:.4f}"
    )

    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Data")
    ax.set_ylabel("Preço")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    plt.tight_layout()
    plt.show()

    print("\n" + "=" * 72)
    print(f"ATIVO: {symbol}")
    print(f"Último fechamento: {last_close:.2f}")
    print(f"GARCH próximo dia   : {row_info['garch_d1_%']:.2f}%")
    print(f"GARCH próxima semana: {row_info['garch_w1_%']:.2f}%")
    print(f"GARCH próximo mês   : {row_info['garch_m1_%']:.2f}%")
    print(f"Score              : {row_info['score']:.4f}")
    print(f"Perfil             : {row_info['perfil']}")
    print("-" * 72)
    print("Pernas da vol prevista da próxima semana:")
    for leg in legs:
        print(
            f"{leg['label']:>4} | "
            f"Movimento: {leg['pct_move']*100:>6.2f}% | "
            f"Lower: {leg['lower']:>10.2f} | "
            f"Upper: {leg['upper']:>10.2f}"
        )
    print("=" * 72 + "\n")

# ============================================================
# 8) MAIN
# ============================================================

def main():
    tv = get_tv()

    print("Montando ranking PRO forward-looking...")
    ranking = build_ranking(tv, UNIVERSE)

    out = ranking.head(TOP_N).copy()
    out["score"] = out["score"].round(4)
    out["last_close"] = out["last_close"].round(2)
    out["garch_d1_%"] = out["garch_d1_%"].round(2)
    out["garch_w1_%"] = out["garch_w1_%"].round(2)
    out["garch_m1_%"] = out["garch_m1_%"].round(2)

    print("\nTOP RANKING PRO - GARCH FORWARD LOOKING")
    print(
        out[["symbol", "score", "perfil", "last_close", "garch_d1_%", "garch_w1_%", "garch_m1_%"]]
        .to_string(index=False)
    )

    if SELECTED_SYMBOL is None:
        chosen = ranking.iloc[0]["symbol"]
    else:
        chosen = SELECTED_SYMBOL

    print(f"\nAtivo escolhido para gráfico: {chosen}")

    row_sel = ranking.loc[ranking["symbol"] == chosen]
    if row_sel.empty:
        raise RuntimeError(f"{chosen} não apareceu no ranking. Verifique símbolo/exchange.")
    row_sel = row_sel.iloc[0]

    df_daily = fetch_data(
        tv, chosen, Interval.in_daily,
        exchange=EXCHANGE, n_bars=N_BARS_DAILY
    )
    if df_daily is None or df_daily.empty:
        raise RuntimeError(f"Não foi possível baixar dados diários de {chosen}")

    df_weekly = fetch_data(
        tv, chosen, Interval.in_weekly,
        exchange=EXCHANGE, n_bars=N_BARS_WEEKLY
    )
    if df_weekly is None or df_weekly.empty:
        raise RuntimeError(f"Não foi possível baixar dados semanais de {chosen}")

    garch_info = fit_garch_multi_horizon(df_daily)
    if garch_info is None:
        raise RuntimeError(f"Não foi possível ajustar GARCH para {chosen}")

    plot_asset_with_forecast(chosen, df_weekly, row_sel, garch_info)

if __name__ == "__main__":
    main()